In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

#Smart file locator
if os.path.exists("data/data/SPY_15m_data.csv"):
    target_file = "data/data/SPY_15m_data.csv"
else:
    target_file = "data/SPY_15m_data.csv"

#Load the data 
data = pd.read_csv(target_file, index_col=0, parse_dates=True)

#Rebuild the core baselines for comparison
data['market_return'] = data['close'].pct_change()
data['buy_and_hold'] = (1 + data['market_return']).cumprod()

#Clean up the first empty row
data.dropna(inplace=True)

print(f"Environment ready! Data loaded and baselines calculated. Total rows: {len(data)}")

Environment ready! Data loaded and baselines calculated. Total rows: 1559


In [11]:
#Parameters
ma_length = 20
stop_loss_pct = 0.005

#Indicator
data['SMA_20'] = data['close'].rolling(window=ma_length).mean()

#Logic and Return
data['raw_signal'] = np.where(data['close'] > data['SMA_20'], 1, -1)
data['raw_strat_return'] = data['raw_signal'].shift(1) * data['market_return']
data['strat_return_with_sl'] = np.clip(data['raw_strat_return'], -stop_loss_pct, np.inf)
data['single_ma_cumulative'] = (1 + data['strat_return_with_sl']).cumprod()

#Results
final_return = (data['single_ma_cumulative'].iloc[-1] - 1) * 100
buy_hold_return = (data['buy_and_hold'].iloc[-1] - 1) * 100

print("--- SINGLE MA (20-Period) RESULTS ---")
print(f"Buy & Hold Baseline: {buy_hold_return:.2f}%")
print(f"Strategy Return (with {stop_loss_pct*100}% Stop-Loss): {final_return:.2f}%")


--- SINGLE MA (20-Period) RESULTS ---
Buy & Hold Baseline: 5.40%
Strategy Return (with 0.5% Stop-Loss): -0.85%


In [13]:
#Parameters
slow_mas_to_test = [50, 100, 200]
stop_loss_pct = 0.005

#We will store the final scores here
results_scoreboard_single = []

#Loop through the slow MAs
for ma in slow_mas_to_test:
    ma_col = f'SMA_{ma}'
    signal_col = f'single_signal_{ma}'
    data[signal_col] = np.where(data['close'] > data[ma_col], 1, -1)
    raw_ret_col = f'raw_single_ret_{ma}'
    data[raw_ret_col] = data[signal_col].shift(1) * data['market_return']
    sl_ret_col = f'sl_single_ret_{ma}'
    data[sl_ret_col] = np.clip(data[raw_ret_col], -stop_loss_pct, np.inf)
    cum_ret_col = f'cum_single_ret_{ma}'
    data[cum_ret_col] = (1 + data[sl_ret_col]).cumprod()
    final_return = (data[cum_ret_col].iloc[-1] - 1) * 100
    results_scoreboard_single.append((ma, final_return))

#Results
print("--- SINGLE SLOW MA RESULTS ---")
print(f"Buy & Hold Baseline: {buy_hold_return:.2f}%\n")

for ma, final_return in results_scoreboard_single:
    print(f"Single {ma}-Period MA: {final_return:.2f}%")

--- SINGLE SLOW MA RESULTS ---
Buy & Hold Baseline: 5.40%

Single 50-Period MA: 5.25%
Single 100-Period MA: 6.67%
Single 200-Period MA: -0.69%


In [15]:
#Parameters
fast_ma = 20
slow_mas_to_test = [50, 100, 200]
stop_loss_pct = 0.005

#Make sure the Fast MA is calculated
data['SMA_20'] = data['close'].rolling(window=fast_ma).mean()

#We will store the final scores here to print at the end
results_scoreboard = []

#Loop through all three slow MAs
for slow_ma in slow_mas_to_test:
    ma_col_name = f'SMA_{slow_ma}'
    data[ma_col_name] = data['close'].rolling(window=slow_ma).mean()
    long_condition = (data['close'] > data['SMA_20']) & (data['close'] > data[ma_col_name])
    short_condition = (data['close'] < data['SMA_20']) & (data['close'] < data[ma_col_name])
    signal_col_name = f'dual_signal_{slow_ma}'
    data[signal_col_name] = np.where(long_condition, 1, np.where(short_condition, -1, 0))
    raw_ret_col = f'raw_ret_{slow_ma}'
    data[raw_ret_col] = data[signal_col_name].shift(1) * data['market_return']
    sl_ret_col = f'sl_ret_{slow_ma}'
    data[sl_ret_col] = np.clip(data[raw_ret_col], -stop_loss_pct, np.inf)
    cum_ret_col = f'cum_ret_{slow_ma}'
    data[cum_ret_col] = (1 + data[sl_ret_col]).cumprod()
    final_return = (data[cum_ret_col].iloc[-1] - 1) * 100
    results_scoreboard.append((slow_ma, final_return))

#Results
buy_hold_return = (data['buy_and_hold'].iloc[-1] - 1) * 100

print("--- DUAL-MA (REGIME + TRIGGER) RESULTS ---")
print(f"Buy & Hold Baseline: {buy_hold_return:.2f}%\n")

for slow_ma, final_return in results_scoreboard:
    print(f"Fast 20 + Slow {slow_ma} Filter: {final_return:.2f}%")

--- DUAL-MA (REGIME + TRIGGER) RESULTS ---
Buy & Hold Baseline: 5.40%

Fast 20 + Slow 50 Filter: 2.51%
Fast 20 + Slow 100 Filter: 3.25%
Fast 20 + Slow 200 Filter: -1.52%
